# CallMeMaybe: señales para revisar la atención

**Brenis Hernández · Proyecto de portafolio**

Esta copia pública conserva el código y las explicaciones. Se retiraron las salidas originales para evitar publicar registros o identificadores; los resultados resumidos se documentan en la ficha del portafolio a partir de la versión original revisada. No se ha vuelto a ejecutar sobre los datos originales. También se omiten mensajes de revisión académica.

**Alcance:** Datos de agosto–noviembre de 2019. Las filas son agregados, no llamadas individuales; las 76 señales corresponden a pares cliente–operador, no necesariamente a 76 personas distintas. Las métricas usadas para formar los grupos no pueden validarlos de forma independiente mediante una comparación de esos mismos indicadores. Los registros sin operador reducen la cobertura de atribución.


# Proyecto Final — CallMeMaybe (Operadores ineficaces)

**PDF del reporte:** https://drive.google.com/file/d/14tXbutWOBxKOFZJpnih_HW0EK66nYsZI/view?usp=sharing

**Dashboard Tableau Public:** https://public.tableau.com/app/profile/brenis.hernandez/viz/Dashboard-Proyectofinal-BrenisHernandez/Dashboard1?publish=yes

## Objetivo
Identificar operadores ineficaces por cliente (user_id) usando:
- tasa de llamadas entrantes perdidas
- tiempo de espera en entrantes
- (si aplica) bajo volumen de llamadas salientes relativo a su cliente


## 2. Datos disponibles

Se utilizarán dos datasets:

- `telecom_dataset_us.csv` (llamadas)
  - user_id, date, direction, internal, operator_id, is_missed_call, calls_count, call_duration, total_call_duration
- `telecom_clients_us.csv` (clientes)
  - user_id, tariff_plan, date_start

**Unidad principal de análisis:** `operator_id` (por cliente `user_id`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy import stats

PATH_CALLS = "/datasets/telecom_dataset_us.csv"
PATH_CLIENTS = "/datasets/telecom_clients_us.csv"

calls = pd.read_csv(PATH_CALLS)
clients = pd.read_csv(PATH_CLIENTS)

print(calls.shape, clients.shape)
calls.head()


El dataset de llamadas contiene actividad para 307 clientes; el dataset de clientes incluye 732 clientes totales, pero no todos aparecen con llamadas en el periodo analizado.

In [ ]:
display(calls.info())
display(clients.info())

print("\nMissing values (calls):")
display(calls.isna().sum())

print("\nMissing values (clients):")
display(clients.isna().sum())

print("\nDuplicates (calls):", calls.duplicated().sum())
print("Duplicates (clients):", clients.duplicated().sum())

# Eliminar duplicados para evitar inflar calls_count y distorsionar métricas
calls = calls.drop_duplicates()

print("Duplicates removed. New shape:", calls.shape)


display(clients.head())
display(calls.head())


Se eliminaron duplicados para evitar inflar calls_count y distorsionar métricas agregadas (tasas y promedios ponderados).

## 3. Preparación y limpieza de datos

En esta sección:
- Convertir tipos de datos (fechas y booleanos)
- Validar valores ausentes / duplicados
- Crear variable clave: **tiempo de espera**
  - `wait_time = total_call_duration - call_duration`
- Crear métricas “por llamada” usando `calls_count` como ponderador (si aplica)


In [ ]:
df = calls.copy()

In [ ]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")
clients["date_start"] = pd.to_datetime(clients["date_start"], errors="coerce")

df["internal"] = df["internal"].replace({"True": True, "False": False})
df["internal"] = df["internal"].fillna(False).astype(bool)

df["is_missed_call"] = df["is_missed_call"].astype(bool)



In [ ]:
df["operator_id"] = df["operator_id"].replace(0, np.nan)


Los valores faltantes en operator_id corresponden a llamadas que no pueden atribuirse a un operador específico (por ejemplo, llamadas sin asignación). Estas filas se mantienen para análisis generales, pero se excluirán del análisis por operador.

In [ ]:

df["wait_time_total"] = df["total_call_duration"] - df["call_duration"]

In [ ]:
df["calls_count"] = pd.to_numeric(df["calls_count"], errors="coerce").fillna(0).astype(int)

df["avg_wait_time"] = np.where(df["calls_count"] > 0, df["wait_time_total"] / df["calls_count"], np.nan)
df["avg_call_duration"] = np.where(df["calls_count"] > 0, df["call_duration"] / df["calls_count"], np.nan)


Los registros parecen estar agregados (una fila puede representar múltiples llamadas), por lo que se calculan promedios ponderados usando calls_count para obtener métricas “por llamada” y evitar sesgo hacia registros con mayor agregación.

In [ ]:
df = df.dropna(subset=["date", "user_id"])


In [ ]:
df_ops = df.dropna(subset=["operator_id"]).copy()
df_ops["operator_id"] = df_ops["operator_id"].astype(int)


Las filas sin operator_id no pueden atribuirse a un operador específico, por lo que se excluyen del análisis por operador para evitar asignaciones incorrectas de desempeño.

## 4. Análisis exploratorio (EDA)

Objetivo del EDA:
- Entender distribución de llamadas por dirección (in/out)
- Proporción de internas vs externas
- Distribución de tiempos de espera
- Llamadas perdidas (missed) y su comportamiento
- Número de operadores y cobertura por cliente


In [ ]:
print("Rango de fechas:", df["date"].min(), "->", df["date"].max())
print("Clientes únicos (user_id):", df["user_id"].nunique())
print("Operadores únicos (operator_id):", df_ops["operator_id"].nunique())


display(df["direction"].value_counts(dropna=False))
display(df["internal"].value_counts(dropna=False))
display(df["is_missed_call"].value_counts(dropna=False))


In [ ]:
# Histograma de avg_wait_time (limitar outliers extremos para ver mejor)
plt.figure(figsize=(8,4))
df["avg_wait_time"].dropna().clip(upper=df["avg_wait_time"].quantile(0.99)).hist(bins=40)
plt.title("Distribución de tiempo de espera promedio por llamada (cap al p99)")
plt.xlabel("Tiempo de espera promedio")
plt.ylabel("Frecuencia")
plt.grid(True, alpha=0.3)
plt.show()




**Interpretación:**
La distribución del tiempo de espera promedio por llamada es asimétrica y presenta una cola larga, lo que indica que existen casos con esperas muy altas.  
Esto sugiere que el tiempo de espera es un KPI útil para diferenciar desempeño, y justifica el uso de percentiles para definir “espera prolongada”.


In [ ]:
# Proporción internas vs externas
internal_share = df.groupby("internal")["calls_count"].sum()
plt.figure(figsize=(5,5))
plt.pie(internal_share, labels=internal_share.index.astype(str), autopct="%1.1f%%")
plt.title("Participación de llamadas internas vs externas (por calls_count)")
plt.show()

**Interpretación:**
La mayoría del volumen de llamadas corresponde a llamadas externas/internas (según se observa en el reparto).  
Esta proporción ayuda a contextualizar la operación y a entender el peso relativo de llamadas internas frente a externas al evaluar eficiencia.


In [ ]:
# Número de llamadas por día (por calls_count)
df["date_day"] = df["date"].dt.tz_convert(None).dt.date if df["date"].dt.tz is not None else df["date"].dt.date

calls_by_day = df.groupby("date_day")["calls_count"].sum()

plt.figure(figsize=(10,4))
calls_by_day.plot()
plt.title("Número de llamadas por día (calls_count)")
plt.xlabel("Fecha")
plt.ylabel("Llamadas")
plt.grid(True, alpha=0.3)
plt.show()


**Interpretación:**
El volumen diario de llamadas muestra variación a lo largo del periodo, lo cual puede reflejar picos de demanda.  
Esta variabilidad es relevante porque niveles de carga distintos pueden afectar tiempos de espera y llamadas perdidas.


In [ ]:
# --- Revisión de outliers (EDA) ---
# Objetivo: detectar valores extremos que podrían distorsionar promedios/tasas.

eda_cols = ["avg_wait_time", "avg_call_duration", "calls_count"]
print(df[eda_cols].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

# Boxplots con cap al p99 para visualizar mejor (sin perder el patrón general)
cap_wait = df["avg_wait_time"].dropna().quantile(0.99)
cap_dur = df["avg_call_duration"].dropna().quantile(0.99)

plt.figure(figsize=(10,4))
plt.boxplot([
    df["avg_wait_time"].dropna().clip(upper=cap_wait),
    df["avg_call_duration"].dropna().clip(upper=cap_dur)
], labels=["avg_wait_time (cap p99)", "avg_call_duration (cap p99)"])
plt.title("Revisión de outliers (cap p99) — espera y duración por llamada")
plt.ylabel("Segundos (aprox.)")
plt.grid(True, alpha=0.3)
plt.show()


**Interpretación (outliers):**  
Se observan valores extremos (outliers) en tiempos de espera y duración por llamada.  
Para evitar que estos casos distorsionen la visualización del comportamiento general, las gráficas del EDA utilizan un *cap* en el percentil 99 (p99).  
En el cálculo de métricas por operador se mantienen los datos completos (sin recorte) y se usan agregaciones/ponderaciones con `calls_count` para reducir el impacto de registros aislados.

**Nota:** Parte de los valores extremos en duración/promedios puede estar influida por llamadas perdidas (call_duration=0).  
Por ello, el KPI principal para eficiencia se centra en **wait time** y **missed rate**, y la duración se usa solo como contexto exploratorio.



In [ ]:
# Distribución de missed rate por operador (solo donde hay llamadas entrantes)
tmp_in = df_ops[df_ops["direction"]=="in"].copy()
tmp_in["missed_calls_count"] = np.where(tmp_in["is_missed_call"], tmp_in["calls_count"], 0)

tmp_rates = (tmp_in.groupby(["user_id","operator_id"])
             .agg(in_calls=("calls_count","sum"),
                  in_missed_calls=("missed_calls_count","sum"))
             .reset_index())

tmp_rates["in_missed_rate"] = np.where(tmp_rates["in_calls"]>0,
                                       tmp_rates["in_missed_calls"]/tmp_rates["in_calls"],
                                       np.nan)

plt.figure(figsize=(8,4))
tmp_rates["in_missed_rate"].dropna().clip(upper=tmp_rates["in_missed_rate"].quantile(0.99)).hist(bins=40)
plt.title("Distribución de tasa de llamadas perdidas por operador (cap p99)")
plt.xlabel("in_missed_rate")
plt.ylabel("Frecuencia")
plt.grid(True, alpha=0.3)
plt.show()


**Interpretación:**  
La tasa de llamadas perdidas por operador presenta alta concentración en valores cercanos a 0 y una cola larga.  
Esto respalda el uso de percentiles (p75) para separar operadores con tasas anormalmente altas.


## 5. Métricas clave por operador

Construiremos métricas por operador y cliente (`operator_id` + `user_id`):

**Entrantes**
- Total de llamadas entrantes
- Total de llamadas entrantes perdidas
- Tasa de perdidas entrantes
- Tiempo promedio de espera en entrantes (ponderado por calls_count)

**Salientes**
- Total de llamadas salientes
- Identificar si el operador es “activo en salientes”
- Comparar el volumen de llamadas salientes respecto a otros operadores del mismo cliente.

In [ ]:
# Separar por dirección usando el dataset SOLO con operator_id válido
df_in = df_ops[df_ops["direction"] == "in"].copy()
df_out = df_ops[df_ops["direction"] == "out"].copy()

print("df_in shape:", df_in.shape)
print("df_out shape:", df_out.shape)
# Entrantes: totales y perdidas (crear conteo de perdidas)
df_in["missed_calls_count"] = np.where(df_in["is_missed_call"], df_in["calls_count"], 0)

in_metrics = (
    df_in.groupby(["user_id", "operator_id"])
    .agg(
        in_calls=("calls_count", "sum"),
        in_missed_calls=("missed_calls_count", "sum"),
        in_wait_total=("wait_time_total", "sum"),
    )
    .reset_index()
)

in_metrics["in_missed_rate"] = np.where(
    in_metrics["in_calls"] > 0,
    in_metrics["in_missed_calls"] / in_metrics["in_calls"],
    np.nan
)

in_metrics["in_avg_wait_time"] = np.where(
    in_metrics["in_calls"] > 0,
    in_metrics["in_wait_total"] / in_metrics["in_calls"],
    np.nan
)


# Salientes: volumen
out_metrics = (
    df_out.groupby(["user_id", "operator_id"], dropna=False)
    .agg(out_calls=("calls_count", "sum"))
    .reset_index()
)

# Merge métricas
metrics = in_metrics.merge(out_metrics, on=["user_id", "operator_id"], how="outer")
metrics["in_calls"] = metrics["in_calls"].fillna(0)
metrics["in_missed_calls"] = metrics["in_missed_calls"].fillna(0)
metrics["out_calls"] = metrics["out_calls"].fillna(0)

# Definir operador "activo" en out si tiene al menos 1 llamada saliente
metrics["out_active"] = metrics["out_calls"] > 0

display(metrics.head())
metrics.describe(include="all")


### Definición de umbrales (percentiles)
Para evitar umbrales arbitrarios, se utilizaron percentiles como referencia de desempeño relativo:
- **p75** para identificar valores “altos” en KPIs negativos (tasa de perdidas y tiempo de espera).
- **p25 por cliente** para identificar operadores con **bajo volumen de llamadas salientes** respecto a sus pares del mismo cliente (solo cuando hay suficientes operadores con actividad saliente).
Este enfoque permite comparar rendimiento dentro del contexto real de la operación y reduce sesgos por diferencias entre clientes.


In [ ]:

# Umbrales globales 
miss_rate_thr = metrics.loc[metrics["in_calls"] > 0, "in_missed_rate"].quantile(0.75)
wait_thr = metrics.loc[metrics["in_calls"] > 0, "in_avg_wait_time"].quantile(0.75)

print("Umbral tasa de perdidas (p75):", miss_rate_thr)
print("Umbral espera promedio (p75):", wait_thr)

# --- Resumen de criterios de ineficacia ---
print("Criterios usados para clasificar operadores:")
print(f"1) high_missed_rate: in_missed_rate >= p75 = {miss_rate_thr:.6f} (solo si in_calls > 0)")
print(f"2) long_wait_time: in_avg_wait_time >= p75 = {wait_thr:.3f} (solo si in_calls > 0)")
print("3) low_out_calls: out_calls <= p25 del cliente (solo si out_active=True y hay >=4 operadores out_active en ese cliente)")
print("Regla final: operador ineficaz si cumple al menos 2 de los 3 criterios.")



In [ ]:


out_active = metrics[metrics["out_active"]].copy()

# contar operadores out_active por cliente
n_out_ops = out_active.groupby("user_id")["operator_id"].transform("count")

out_active["out_calls_p25_by_client"] = np.where(
    n_out_ops >= 4,
    out_active.groupby("user_id")["out_calls"].transform(lambda s: s.quantile(0.25)),
    np.nan
)

# Pasar la columna al df principal
metrics = metrics.merge(
    out_active[["user_id", "operator_id", "out_calls_p25_by_client"]],
    on=["user_id", "operator_id"],
    how="left"
)

# low_out_calls solo aplica si:
# - el operador tiene llamadas salientes (out_active)
# - existe umbral p25 en su cliente (no NaN)
# - y el operador tiene al menos 1 llamada entrante (para que el criterio no "domine" solo)
metrics["low_out_calls"] = np.where(
    (metrics["out_active"]) &
    (metrics["out_calls_p25_by_client"].notna()) &
    (metrics["in_calls"] > 0),
    metrics["out_calls"] <= metrics["out_calls_p25_by_client"],
    False
)

# Criterios entrantes
metrics["high_missed_rate"] = np.where(
    metrics["in_calls"] > 0,
    metrics["in_missed_rate"] >= miss_rate_thr,
    False
)

metrics["long_wait_time"] = np.where(
    metrics["in_calls"] > 0,
    metrics["in_avg_wait_time"] >= wait_thr,
    False
)

# Reglas: ineficaz si cumple >=2 criterios
criteria_cols = ["high_missed_rate", "long_wait_time", "low_out_calls"]
metrics["n_criteria_met"] = metrics[criteria_cols].sum(axis=1)

metrics["is_ineffective"] = metrics["n_criteria_met"] >= 2

display(metrics[[
    "user_id", "operator_id", "in_calls", "in_missed_rate", "in_avg_wait_time", "out_calls",
    "high_missed_rate", "long_wait_time", "low_out_calls", "n_criteria_met", "is_ineffective"
]].head(10))

print("Operadores ineficaces:", metrics["is_ineffective"].sum())



## 6. Lista de operadores ineficaces

A continuación se muestra la lista de operadores marcados como ineficaces con las métricas que justifican su clasificación.


In [ ]:
ineffective_ops = (
    metrics[metrics["is_ineffective"]]
    .sort_values(["n_criteria_met", "in_missed_rate", "in_avg_wait_time", "out_calls"], ascending=[False, False, False, True])
)

display(ineffective_ops.head(20))


## 7. Pruebas de hipótesis

Se probarán hipótesis para validar si los operadores ineficaces difieren significativamente de los eficaces.

**H1 (espera):**
- H0: no hay diferencia en el tiempo de espera promedio (in_avg_wait_time) entre operadores eficaces e ineficaces
- H1: sí hay diferencia

**H2 (llamadas perdidas):**
- H0: la tasa de llamadas perdidas (in_missed_rate) es igual entre eficaces e ineficaces
- H1: es diferente

**H3 (salientes, solo operadores activos en out):**
- H0: no hay diferencia en out_calls entre eficaces e ineficaces
- H1: sí hay diferencia

Nivel de significancia: α = 0.05


In [ ]:
alpha = 0.05

# Separar grupos
eff = metrics[~metrics["is_ineffective"]].copy()
ineff = metrics[metrics["is_ineffective"]].copy()

# --- H1: espera (Mann-Whitney, robusto ante no-normalidad) ---
x1 = eff.loc[eff["in_calls"] > 0, "in_avg_wait_time"].dropna()
x2 = ineff.loc[ineff["in_calls"] > 0, "in_avg_wait_time"].dropna()

if len(x1) > 0 and len(x2) > 0:
    stat, p = stats.mannwhitneyu(x1, x2, alternative="two-sided")
    print("H1 Espera (Mann-Whitney): p-value =", p, "| Rechazar H0:", p < alpha)
else:
    print("H1: No hay datos suficientes para comparar.")

# --- H2: tasa de perdidas (Mann-Whitney sobre tasas) ---
y1 = eff.loc[eff["in_calls"] > 0, "in_missed_rate"].dropna()
y2 = ineff.loc[ineff["in_calls"] > 0, "in_missed_rate"].dropna()

if len(y1) > 0 and len(y2) > 0:
    stat, p = stats.mannwhitneyu(y1, y2, alternative="two-sided")
    print("H2 Perdidas (Mann-Whitney): p-value =", p, "| Rechazar H0:", p < alpha)
else:
    print("H2: No hay datos suficientes para comparar.")

# --- H3: salientes (solo out_active) ---
z1 = eff.loc[eff["out_active"], "out_calls"].dropna()
z2 = ineff.loc[ineff["out_active"], "out_calls"].dropna()

if len(z1) > 0 and len(z2) > 0:
    stat, p = stats.mannwhitneyu(z1, z2, alternative="two-sided")
    print("H3 Salientes (Mann-Whitney): p-value =", p, "| Rechazar H0:", p < alpha)
else:
    print("H3: No hay datos suficientes para comparar (pocos out_active).")


### Comparación visual: tiempo de espera — operadores eficaces vs ineficaces

A continuación se compara el tiempo promedio de espera en llamadas entrantes entre operadores clasificados como eficaces e ineficaces.  
Esto permite visualizar si la diferencia encontrada en la prueba estadística también se observa claramente en la distribución de los datos.


In [ ]:
plt.figure(figsize=(7,4))

data = [
    eff.loc[eff["in_calls"] > 0, "in_avg_wait_time"].dropna(),
    ineff.loc[ineff["in_calls"] > 0, "in_avg_wait_time"].dropna()
]

plt.boxplot(data, labels=["Eficaces", "Ineficaces"])
plt.title("Tiempo promedio de espera — Eficaces vs Ineficaces")
plt.ylabel("Segundos")
plt.grid(True, alpha=0.3)
plt.show()


### Comparación visual: tasa de llamadas perdidas

Se muestra la distribución de la tasa de llamadas perdidas en operadores eficaces e ineficaces.


In [ ]:
plt.figure(figsize=(7,4))

data = [
    eff.loc[eff["in_calls"] > 0, "in_missed_rate"].dropna(),
    ineff.loc[ineff["in_calls"] > 0, "in_missed_rate"].dropna()
]

plt.boxplot(data, labels=["Eficaces", "Ineficaces"])
plt.title("Tasa de llamadas perdidas — Eficaces vs Ineficaces")
plt.ylabel("Proporción")
plt.grid(True, alpha=0.3)
plt.show()


### Comparación visual: volumen de llamadas salientes (operadores activos)

Solo se consideran operadores que realizan llamadas salientes.


In [ ]:
plt.figure(figsize=(7,4))

data = [
    eff.loc[eff["out_active"], "out_calls"].dropna(),
    ineff.loc[ineff["out_active"], "out_calls"].dropna()
]

plt.boxplot(data, labels=["Eficaces", "Ineficaces"])
plt.title("Volumen de llamadas salientes — Eficaces vs Ineficaces")
plt.ylabel("Cantidad de llamadas")
plt.grid(True, alpha=0.3)
plt.show()


## 8. Integración con datos de clientes

Se analizará si existe alguna relación entre:
- Plan tarifario (`tariff_plan`)
- Antigüedad del cliente (`date_start`)
y la presencia de operadores ineficaces.


In [ ]:
metrics_clients = metrics.merge(clients, on="user_id", how="left")

# --- Dataset final para Tableau / reporte ---
final_ops = metrics.merge(clients, on="user_id", how="left")

# Columnas recomendadas para dashboard
final_ops = final_ops[[
    "user_id", "operator_id", "tariff_plan", "date_start",
    "in_calls", "in_missed_calls", "in_missed_rate", "in_avg_wait_time",
    "out_calls", "out_active",
    "out_calls_p25_by_client", "low_out_calls",
    "high_missed_rate", "long_wait_time",
    "n_criteria_met", "is_ineffective"
]].copy()

final_ops.head()

final_ops.to_csv("callmemaybe_operators_dashboard.csv", index=False)
print("Archivo exportado: callmemaybe_operators_dashboard.csv")


# Ineficaces por plan
ineff_by_plan = (
    metrics_clients.groupby("tariff_plan")["is_ineffective"]
    .mean()
    .sort_values(ascending=False)
    .to_frame("share_ineffective")
)

display(ineff_by_plan)

# Conteo absoluto por plan
count_by_plan = metrics_clients.groupby("tariff_plan")["is_ineffective"].agg(["count","sum"])
count_by_plan = count_by_plan.rename(columns={"sum":"ineffective_count"})
display(count_by_plan)


## 9. Fuentes consultadas 

pandas — GroupBy / Aggregation: para construir métricas por operador (sumas, promedios y tasas).
https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html

pandas — merge (joins): para integrar información de clientes (tariff_plan).
https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html

pandas — to_datetime: para convertir columnas de fecha y agrupar por día.
https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html

NumPy — where: para crear columnas condicionales (p.ej., missed_calls_count).
https://numpy.org/doc/stable/reference/generated/numpy.where.html

Matplotlib — histogram: para graficar distribución de tiempos/duraciones en el EDA.
https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.hist.html

SciPy — mannwhitneyu: para comparar grupos (ineficaces vs eficaces) sin asumir normalidad.
https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.mannwhitneyu.html

ICMI — Average Speed of Answer (ASA): definición de ASA (tiempo promedio para contestar una llamada).

Five9 — Abandoned Call / Call Abandonment Rate: definición de llamada abandonada y métrica de abandono.

Percentiles y cuartiles (definición y uso): soporte para umbrales p75/p25.
https://en.wikipedia.org/wiki/Quantile


## 10. Conclusiones y recomendaciones

### Conclusiones principales
A partir del análisis de los datos de CallMeMaybe, se construyeron métricas a nivel de **operador por cliente** (`operator_id` + `user_id`) para evaluar desempeño operativo. Las variables clave fueron:

- **Tasa de llamadas entrantes perdidas** (`in_missed_rate`)
- **Tiempo promedio de espera en llamadas entrantes** (`in_avg_wait_time`)
- **Volumen de llamadas salientes** (`out_calls`) cuando el operador participa en salientes

Para la clasificación se usaron umbrales basados en distribución:
- **p75** para detectar valores altos en KPIs negativos (perdidas y espera).
- **p25 por cliente** para detectar bajo desempeño relativo en llamadas salientes (cuando aplica).

Un operador se marcó como **ineficaz** cuando cumple **al menos 2** de los criterios definidos.

Además, las pruebas estadísticas (Mann–Whitney) confirmaron que las diferencias entre operadores eficaces e ineficaces en:
- tiempo de espera,
- tasa de llamadas perdidas,
- y volumen de llamadas salientes (cuando aplica),
son **estadísticamente significativas** en las comparaciones guardadas. Sin embargo, los grupos se formaron usando esos mismos indicadores; estas pruebas describen la separación y no constituyen una validación independiente de la clasificación.

### Implicaciones para supervisión
Los operadores ineficaces pueden estar generando:
- mayor tiempo de espera para clientes,
- mayor proporción de llamadas no atendidas,
- potencial pérdida de oportunidades de servicio/venta,
- y sobrecarga para operadores con mejor desempeño.

El enfoque basado en percentiles permite evaluar rendimiento de forma **relativa** al contexto operativo, evitando umbrales fijos poco realistas.

### Recomendaciones accionables
1. **Revisar asignación de llamadas**  
   Identificar si existe desbalance en distribución (por tipo de llamada, horarios o carga operativa).

2. **Capacitación y acompañamiento**  
   Diseñar planes específicos para operadores con alta tasa de perdidas y/o alta espera.

3. **Monitoreo de llamadas salientes (si aplica)**  
   Definir objetivos mínimos y monitorear `out_calls` dentro de cada cliente.

4. **Seguimiento periódico**  
   Repetir el análisis de forma mensual/semanal para observar evolución e impacto de acciones.

### Limitaciones
- La definición depende de percentiles del periodo observado; puede variar en otros periodos.
- Los registros parecen agregados por día, por lo que se usaron ponderaciones con `calls_count`.
- No se cuenta con variables de complejidad de llamada o características del operador que podrían explicar parte del desempeño.

